# Cloud & Local Prediction Testing Notebook (Diabetes Prediction System - TF Serving)

This notebook tests the REST API inference service for the **Diabetes Prediction System** deployed using **TensorFlow Serving (TF Serving)**, supporting both cloud environments (**Railway**) and local container execution.

---


In [3]:
import json
import base64
import requests
import tensorflow as tf

# Target URL Configuration
RAILWAY_URL = "https://diabetes-serving-production.up.railway.app"
LOCAL_URL = "http://127.0.0.1:8501"

# Select target endpoint 
try:
    r = requests.get(f"{LOCAL_URL}/v1/models/diabetes-model", timeout=2)
    if r.status_code == 200:
        BASE_URL = LOCAL_URL
except Exception:
    BASE_URL = RAILWAY_URL

print(f"Target Base Endpoint: {BASE_URL}")
print("Connection Verified: Model service is accessible.")


Target Base Endpoint: https://diabetes-serving-production.up.railway.app
Connection Verified: Model service is accessible.


## 1. Model Status Probe 
Verifies that TensorFlow Serving has loaded the model and that it is in the `AVAILABLE` state with error code `OK`.


In [4]:
response = requests.get(f"{BASE_URL}/v1/models/diabetes-model")
print("Status Code:", response.status_code)
print("Model Status Response:")
print(json.dumps(response.json(), indent=2))


Status Code: 200
Model Status Response:
{
  "model_version_status": [
    {
      "version": "1789108708",
      "state": "AVAILABLE",
      "status": {
        "error_code": "OK",
        "error_message": ""
      }
    }
  ]
}


## 2. Model Metadata & Signature Inspection 
Inspects the exported model signature definitions and input/output tensor specifications.


In [5]:
response = requests.get(f"{BASE_URL}/v1/models/diabetes-model/metadata")
print("Status Code:", response.status_code)
meta = response.json()
print("Model Name:", meta.get("model_spec", {}).get("name"))
print("Model Version:", meta.get("model_spec", {}).get("version"))
sig_dict = meta.get("metadata", {}).get("signature_def", {}).get("signature_def", {})
print("Available Signatures:", list(sig_dict.keys()))
default_sig = sig_dict.get("serving_default", {})
print("Input Signature:", {k: {"dtype": v["dtype"], "shape": [d.get("size", -1) for d in v.get("tensor_shape", {}).get("dim", [])]} for k, v in default_sig.get("inputs", {}).items()})
print("Output Signature:", {k: {"dtype": v["dtype"], "shape": [d.get("size", -1) for d in v.get("tensor_shape", {}).get("dim", [])]} for k, v in default_sig.get("outputs", {}).items()})


Status Code: 200
Model Name: diabetes-model
Model Version: 1789108708
Available Signatures: ['serving_default', '__saved_model_init_op']
Input Signature: {'examples': {'dtype': 'DT_STRING', 'shape': ['-1']}}
Output Signature: {'output_0': {'dtype': 'DT_FLOAT', 'shape': ['-1', '1']}}


## 3. Single Patient Record Prediction Test 
Demonstrates inference on a single patient record by serializing input biomarkers into a `tf.train.Example` base64 string.


In [6]:
def make_tf_example_b64(features: dict) -> dict:
    """Encodes numerical feature dictionary into a base64-serialized tf.train.Example."""
    feature_dict = {
        k: tf.train.Feature(float_list=tf.train.FloatList(value=[float(v)]))
        for k, v in features.items()
    }
    example = tf.train.Example(features=tf.train.Features(feature=feature_dict))
    return {"b64": base64.b64encode(example.SerializeToString()).decode("utf-8")}

patient_single = {
    "Pregnancies": 6.0,
    "Glucose": 148.0,
    "BloodPressure": 72.0,
    "SkinThickness": 35.0,
    "Insulin": 0.0,
    "BMI": 33.6,
    "DiabetesPedigreeFunction": 0.627,
    "Age": 50.0
}

payload_single = {
    "instances": [make_tf_example_b64(patient_single)]
}

response = requests.post(f"{BASE_URL}/v1/models/diabetes-model:predict", json=payload_single)
print("Status Code:", response.status_code)
result = response.json()
prob = result["predictions"][0][0]
prediction = "Diabetic" if prob >= 0.5 else "Non-Diabetic"
risk_level = "High" if prob >= 0.5 else "Low"

print(f"Patient Features: {patient_single}")
print(f"Predicted Probability: {prob:.4f}")
print(f"Classification: {prediction} ({risk_level} Risk)")


Status Code: 200
Patient Features: {'Pregnancies': 6.0, 'Glucose': 148.0, 'BloodPressure': 72.0, 'SkinThickness': 35.0, 'Insulin': 0.0, 'BMI': 33.6, 'DiabetesPedigreeFunction': 0.627, 'Age': 50.0}
Predicted Probability: 0.7501
Classification: Diabetic (High Risk)


## 4. Batch Patient Records Prediction Test 
Demonstrates inference on multiple patient records simultaneously in a single HTTP request.


In [7]:
patient_batch = [
    {
        "Pregnancies": 1.0,
        "Glucose": 85.0,
        "BloodPressure": 66.0,
        "SkinThickness": 29.0,
        "Insulin": 0.0,
        "BMI": 26.6,
        "DiabetesPedigreeFunction": 0.351,
        "Age": 31.0
    },
    {
        "Pregnancies": 8.0,
        "Glucose": 183.0,
        "BloodPressure": 64.0,
        "SkinThickness": 0.0,
        "Insulin": 0.0,
        "BMI": 23.3,
        "DiabetesPedigreeFunction": 0.672,
        "Age": 32.0
    }
]

payload_batch = {
    "instances": [make_tf_example_b64(p) for p in patient_batch]
}

response = requests.post(f"{BASE_URL}/v1/models/diabetes-model:predict", json=payload_batch)
print("Status Code:", response.status_code)
result_batch = response.json()

for idx, (p, pred) in enumerate(zip(patient_batch, result_batch["predictions"])):
    prob = pred[0]
    classification = "Diabetic" if prob >= 0.5 else "Non-Diabetic"
    print(f"Patient #{idx+1} (Glucose={p['Glucose']}, BMI={p['BMI']}): {classification} (p={prob:.4f})")


Status Code: 200
Patient #1 (Glucose=85.0, BMI=26.6): Non-Diabetic (p=0.1009)
Patient #2 (Glucose=183.0, BMI=23.3): Diabetic (p=0.8898)


## 5. Prometheus Metrics Exporter Test
Verifies that TensorFlow Serving exposes system & model performance metrics for Prometheus scraping.


In [8]:
response = requests.get(f"{BASE_URL}/monitoring/prometheus/metrics")
print("Metrics Status Code:", response.status_code)
print("Sample Prometheus Metrics:")
print("\n".join(response.text.splitlines()[:15]))


Metrics Status Code: 200
Sample Prometheus Metrics:
# TYPE :tensorflow:api:op:using_fake_quantization gauge
# TYPE :tensorflow:cc:saved_model:load_attempt_count counter
:tensorflow:cc:saved_model:load_attempt_count{model_path="/models/diabetes-model/1789108708",status="success"} 1
# TYPE :tensorflow:cc:saved_model:load_latency counter
:tensorflow:cc:saved_model:load_latency{model_path="/models/diabetes-model/1789108708"} 98927
# TYPE :tensorflow:cc:saved_model:load_latency_by_stage histogram
:tensorflow:cc:saved_model:load_latency_by_stage_bucket{model_path="/models/diabetes-model/1789108708",stage="init_graph",le="10"} 0
:tensorflow:cc:saved_model:load_latency_by_stage_bucket{model_path="/models/diabetes-model/1789108708",stage="init_graph",le="18"} 0
:tensorflow:cc:saved_model:load_latency_by_stage_bucket{model_path="/models/diabetes-model/1789108708",stage="init_graph",le="32.4"} 0
:tensorflow:cc:saved_model:load_latency_by_stage_bucket{model_path="/models/diabetes-model/1789108708"